# Post-generation rewriting: Colab runs

Draft, then rewrite toward a target trait and level, then blind extraction, then
`score_response`. Llama-3.1-8B-Instruct in 4-bit NF4, greedy decoding,
seed 20261026, on an L4 GPU. Records are written to Google Drive and copied to
`artifacts/rewriting/`.

Greedy outputs depend on batch composition, so rerun with the same `traits`
and `batch_size` to reproduce the records exactly.

## Setup
Clone or update the branch, mount Drive, and load the model once. `HF_TOKEN` is a Colab secret.

In [ ]:
import os
os.chdir("/content")
if not os.path.exists("llm-personality-conditioning"):
    !git clone -b post-generation-rewriting https://github.com/beedle2017/llm-personality-conditioning.git
%cd /content/llm-personality-conditioning
!git pull
!pip install -q bitsandbytes

Cloning into 'llm-personality-conditioning'...
remote: Enumerating objects: 364, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (154/154), done.
remote: Total 364 (delta 65), reused 132 (delta 32), pack-reused 174 (from 1)
Receiving objects: 100% (364/364), 100.17 MiB | 20.45 MiB/s, done.
Resolving deltas: 100% (108/108), done.
/content/llm-personality-conditioning
Already up to date.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/rewriting

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
import sys
sys.path[:0] = ["/content/llm-personality-conditioning/src",
                "/content/llm-personality-conditioning/scripts"]

from rewriting.llm import HFGenerator
import run_rewriting

generator = HFGenerator.load(load_4bit=True)
OUT = "/content/drive/MyDrive/rewriting"

## Timing run
Three Agreeableness items at levels 1 and 5, to estimate run time.

In [ ]:
!time python scripts/run_rewriting.py --traits A --limit 3 --levels 1 5 --batch-size 8 --load-4bit \
    --out /content/drive/MyDrive/rewriting/timing.jsonl

Loading weights: 100% 291/291 [00:04<00:00, 65.24it/s]
seed 20261026: 24 records

real	0m40.530s
user	0m38.711s
sys	0m3.321s


In [ ]:
!python scripts/rewriting_report.py /content/drive/MyDrive/rewriting/timing.jsonl

trait condition level  n   rate  PIS  invalid flip_up flip_down
A     draft       -    3  100.0      -    0.0    0.0    0.0
A     free        1    3   33.3   66.7    0.0    0.0   66.7
A     free        5    3  100.0    0.0    0.0    0.0    0.0
A     locked      1    3  100.0   -0.0    0.0    0.0    0.0
A     locked      5    3  100.0    0.0    0.0    0.0    0.0
A     no_draft    1    3   66.7   33.3    0.0    0.0   33.3
A     no_draft    5    3  100.0    0.0    0.0    0.0    0.0
A     paraphrase  -    3  100.0      -    0.0    0.0    0.0

condition  length_ratio  word_jaccard  numbers_kept  refusal %
free             1.12         0.33            -       0.0
locked           1.02         0.30            -       0.0
no_draft         0.97         0.26            -       0.0
paraphrase       1.01         0.51            -       0.0


## Pilot
Ten items per trait at levels 1 and 5. This run is also the source of the extractor label sample.

In [ ]:
!python scripts/run_rewriting.py --limit 10 --levels 1 5 --load-4bit --batch-size 16 \
    --out /content/drive/MyDrive/rewriting/pilot.jsonl
!python scripts/rewriting_report.py /content/drive/MyDrive/rewriting/pilot.jsonl

Loading weights: 100% 291/291 [00:04<00:00, 65.18it/s]
seed 20261026: 400 records
trait condition level  n   rate  PIS  invalid flip_up flip_down
A     draft       -   10  100.0      -    0.0    0.0    0.0
A     free        1   10   50.0   50.0    0.0    0.0   50.0
A     free        5   10  100.0    0.0    0.0    0.0    0.0
A     locked      1   10  100.0   -0.0    0.0    0.0    0.0
A     locked      5   10  100.0    0.0    0.0    0.0    0.0
A     no_draft    1   10   60.0   40.0    0.0    0.0   40.0
A     no_draft    5   10  100.0    0.0    0.0    0.0    0.0
A     paraphrase  -   10  100.0      -    0.0    0.0    0.0
C     draft       -   10   40.0      -    0.0    0.0    0.0
C     free        1   10   20.0   20.0    0.0    0.0   20.0
C     free        5   10   60.0   20.0    0.0   20.0    0.0
C     locked      1   10   50.0  -10.0    0.0   20.0   10.0
C     locked      5   10   50.0   10.0    0.0   10.0    0.0
C     no_draft    1   10   10.0   30.0    0.0    0.0   30.0
C     no_draft

## Extractor label sample
60 statements without the extractor's answer. Labels and the agreement check are in `artifacts/rewriting/extractor_check/`.

In [ ]:
!python scripts/rewriting_report.py {OUT}/pilot.jsonl --label-sample 60 --labels {OUT}/labels.csv

wrote 60 rows to /content/drive/MyDrive/rewriting/labels.csv; fill human_label, then rerun with --labels only


## Check of the hesitant N items
Rerun after scoping the "more hesitant" extraction instruction. On `bb-N-011-01` the extractor should report the option the speaker avoids (Country X).

In [ ]:
run_rewriting.run_experiment(generator, f"{OUT}/check_N.jsonl", traits=["N"], limit=10, levels=[1, 5], batch_size=16)
import json
for r in map(json.loads, open(f"{OUT}/check_N.jsonl")):
    if r["item_id"] == "bb-N-011-01":
        print(r["condition"], r["level"], "->", r["extracted"])

seed 20261026: 80 records
draft None -> Country X.
paraphrase None -> Country X
free 1 -> Country X
free 5 -> Country X
locked 1 -> Country X.
locked 5 -> Country X.
no_draft 1 -> Country X
no_draft 5 -> Country X.


## Full development run
All 340 development items, every condition, levels 1-5, one trait at a time. About 2 hours on the L4.

In [ ]:
for t in ["O", "C", "E", "A", "N"]:
    run_rewriting.run_experiment(generator, f"{OUT}/dev_{t}.jsonl", traits=[t], batch_size=16)

seed 20261026: 1190 records
seed 20261026: 1190 records
seed 20261026: 1360 records
seed 20261026: 1190 records
seed 20261026: 850 records


In [ ]:
!cat {OUT}/dev_*.jsonl > {OUT}/dev_all.jsonl
!python scripts/rewriting_report.py {OUT}/dev_all.jsonl

trait condition level  n   rate  PIS  invalid flip_up flip_down
A     draft       -   70   98.6      -    0.0    0.0    0.0
A     free        1   70   64.3   34.3    0.0    0.0   34.3
A     free        2   70   82.6   16.0    1.4    1.4   17.4
A     free        3   70   91.3      -    1.4    1.4    8.7
A     free        4   70   91.3   -7.3    1.4    0.0    7.2
A     free        5   70   98.6    0.0    0.0    1.4    1.4
A     locked      1   70  100.0   -1.4    0.0    1.4    0.0
A     locked      2   70  100.0   -1.4    0.0    1.4    0.0
A     locked      3   70  100.0      -    0.0    1.4    0.0
A     locked      4   70   98.6    0.0    0.0    1.4    1.4
A     locked      5   70  100.0    1.4    0.0    1.4    0.0
A     no_draft    1   70   54.3   44.3    0.0    0.0   44.3
A     no_draft    2   70   81.4   17.1    0.0    0.0   17.1
A     no_draft    3   70   98.6      -    0.0    1.4    1.4
A     no_draft    4   70   98.6    0.0    0.0    1.4    1.4
A     no_draft    5   70   98.6    0

## Openness rerun after a description change

The first run's Openness description ended with "and quick to question convention". At high levels the
rewrites questioned the draft's own decision and hedged (`free` level 5: PIS -19). The phrase was removed
on development (commit b57c265) and Openness was rerun. On Drive this file is `dev_O_v2.jsonl`; in the repo it is
`artifacts/rewriting/dev_O.jsonl`, and the first run is kept as `dev_O_v1.jsonl`. Drafts are identical in
both runs.

In [ ]:
run_rewriting.run_experiment(generator, f"{OUT}/dev_O_v2.jsonl", traits=["O"], batch_size=16)
!python scripts/rewriting_report.py {OUT}/dev_O_v2.jsonl

seed 20261026: 1190 records
trait condition level  n   rate  PIS  invalid flip_up flip_down
O     draft       -   70   67.1      -    0.0    0.0    0.0
O     free        1   70   54.3   12.9    0.0    1.4   14.3
O     free        2   70   54.4   12.7    2.9    0.0   11.8
O     free        3   70   62.9      -    0.0    2.9    7.1
O     free        4   70   67.1    0.0    0.0    2.9    2.9
O     free        5   70   60.9   -6.3    1.4    1.4    7.2
O     locked      1   70   62.9    4.3    0.0    0.0    4.3
O     locked      2   70   65.7    1.4    0.0    0.0    1.4
O     locked      3   70   65.7      -    0.0    0.0    1.4
O     locked      4   70   65.7   -1.4    0.0    0.0    1.4
O     locked      5   70   67.1    0.0    0.0    0.0    0.0
O     no_draft    1   70   50.0   17.1    0.0    2.9   20.0
O     no_draft    2   70   55.7   11.4    0.0    2.9   14.3
O     no_draft    3   70   71.4      -    0.0    8.6    4.3
O     no_draft    4   70   75.7    8.6    0.0    8.6    0.0
O     no